In [ ]:
!pip install opencv-python
!pip install albumentations
!pip install scikit-learn
!pip install tqdm
!pip install pandas

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split

import albumentations as A

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content"

RAW_DIR = os.path.join(
    BASE_DIR,
    "raw_faces"
)

PROCESSED_DIR = (
    "/content/drive/MyDrive/processed_faces"
)

os.makedirs(
    RAW_DIR,
    exist_ok=True
)

os.makedirs(
    PROCESSED_DIR,
    exist_ok=True
)

print(PROCESSED_DIR)

/content/drive/MyDrive/processed_faces


In [ ]:
lfw = fetch_lfw_people(
    min_faces_per_person=5,
    resize=1.0,
    color=True
)

print(lfw.images.shape)

print(
    "Classes:",
    len(lfw.target_names)
)

(5985, 125, 94, 3)
Classes: 423


In [ ]:
target_names = lfw.target_names

for idx,img in enumerate(
    lfw.images
):

    label = target_names[
        lfw.target[idx]
    ]

    label_dir = os.path.join(
        RAW_DIR,
        label
    )

    os.makedirs(
        label_dir,
        exist_ok=True
    )

    image = cv2.cvtColor(
        img.astype(np.uint8),
        cv2.COLOR_RGB2BGR
    )

    cv2.imwrite(
        os.path.join(
            label_dir,
            f"{idx}.jpg"
        ),
        image
    )

print("Raw Dataset Saved")

Raw Dataset Saved


In [ ]:
persons = sorted(
    os.listdir(RAW_DIR)
)

print(
    "Persons:",
    len(persons)
)

Persons: 423


In [ ]:
def enhance_face(face):

    lab = cv2.cvtColor(
        face,
        cv2.COLOR_BGR2LAB
    )

    l,a,b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    cl = clahe.apply(l)

    merged = cv2.merge(
        (cl,a,b)
    )

    return cv2.cvtColor(
        merged,
        cv2.COLOR_LAB2BGR
    )

In [ ]:
def resize_face(face):

    return cv2.resize(
        face,
        (112,112)
    )

In [ ]:
for person in tqdm(persons):

    source_folder = os.path.join(
        RAW_DIR,
        person
    )

    save_folder = os.path.join(
        PROCESSED_DIR,
        person
    )

    os.makedirs(
        save_folder,
        exist_ok=True
    )

    counter = 0

    for image_name in os.listdir(
        source_folder
    ):

        path = os.path.join(
            source_folder,
            image_name
        )

        image = cv2.imread(path)

        if image is None:
            continue

        face = enhance_face(
            image
        )

        face = resize_face(
            face
        )

        cv2.imwrite(
            os.path.join(
                save_folder,
                f"{counter}.jpg"
            ),
            face
        )

        counter += 1

100%|██████████| 423/423 [01:18<00:00,  5.36it/s]


In [ ]:
total = 0

classes = 0

for person in os.listdir(
    PROCESSED_DIR
):

    folder = os.path.join(
        PROCESSED_DIR,
        person
    )

    if os.path.isdir(folder):

        classes += 1

        total += len(
            os.listdir(folder)
        )

print(
    "Classes:",
    classes
)
print(
    "Images:",
    total
)

Classes: 423
Images: 5985


In [ ]:
data = []

persons = sorted(
    os.listdir(PROCESSED_DIR)
)

for person in persons:

    folder = os.path.join(
        PROCESSED_DIR,
        person
    )

    if not os.path.isdir(folder):
        continue

    for img in os.listdir(folder):

        path = os.path.join(
            folder,
            img
        )

        if os.path.isfile(path):

            data.append([
                path,
                person
            ])

df = pd.DataFrame(
    data,
    columns=[
        "image_path",
        "label"
    ]
)

print(df.shape)

print(
    df["label"].nunique()
)

(5985, 2)
423


In [ ]:
MAX_IMAGES_PER_CLASS = 30

balanced = []

for person in df["label"].unique():

    temp = df[df["label"] == person]

    if len(temp) > MAX_IMAGES_PER_CLASS:

        temp = temp.sample(
            MAX_IMAGES_PER_CLASS,
            random_state=42
        )

    balanced.append(temp)

df = pd.concat(
    balanced,
    ignore_index=True
)

print(df.shape)

print(
    df["label"].nunique()
)

(4635, 2)
423


In [ ]:
counts = df["label"].value_counts()

print(counts.describe())

count    423.000000
mean      10.957447
std        7.630885
min        5.000000
25%        5.000000
50%        8.000000
75%       13.500000
max       30.000000
Name: count, dtype: float64


In [ ]:
labels = sorted(
    df["label"].unique()
)

label_map = {
    label:i
    for i,label in enumerate(
        labels
    )
}

df["label_id"] = df[
    "label"
].map(label_map)

In [ ]:
train_df,val_df = train_test_split(

    df,

    test_size=0.2,

    stratify=df["label_id"],

    random_state=42

)

print(train_df.shape)

print(val_df.shape)

(3708, 3)
(927, 3)


In [ ]:
train_df.to_csv(
    "/content/drive/MyDrive/train.csv",
    index=False
)

val_df.to_csv(
    "/content/drive/MyDrive/val.csv",
    index=False
)

pd.DataFrame(
    list(label_map.items()),
    columns=[
        "label",
        "label_id"
    ]
).to_csv(
    "/content/drive/MyDrive/label_map.csv",
    index=False
)

print("Saved")

Saved


In [ ]:
bad = train_df[
    ~train_df["image_path"].apply(
        os.path.exists
    )
]

print(
    "Bad Files:",
    len(bad)
)

Bad Files: 0


In [ ]:
print(train_df["label_id"].nunique())
print(train_df["label_id"].min())
print(train_df["label_id"].max())

423
0
422


In [ ]:
print(train_df["label_id"].value_counts().head(20))

label_id
363    24
12     24
27     24
209    24
155    24
320    24
243    24
29     24
124    24
149    24
186    24
120    24
222    24
9      24
166    24
138    24
391    24
79     24
131    24
17     24
Name: count, dtype: int64


In [ ]:
print(train_df["label_id"].value_counts().describe())

count    423.000000
mean       8.765957
std        6.088959
min        4.000000
25%        4.000000
50%        6.000000
75%       10.500000
max       24.000000
Name: count, dtype: float64
